# MedLift-3D — Kaggle pipeline

End-to-end run on a Kaggle notebook. Built around three Kaggle facts:

* **12-hour session cap** — training uses `--max-hours 11`, checkpoints, and
  resumes automatically when you rerun the same cell in a new session.
* **No internet by default** — the projector is pure PyTorch, so nothing beyond
  Kaggle's preinstalled stack is needed. Logging is CSV on disk, not wandb.
* **`/kaggle/working` is the only writable path** — auto-detected.

Enable **GPU (T4 x2 or P100)** in Settings → Accelerator before running.

> Order matters: run the gates first. Four failure modes in this problem are
> silent — they produce plausible loss curves and wrong reconstructions.

## 0 · Setup

Point `REPO` at the code. Either add this repository as a Kaggle *dataset* /
*GitHub* source, or clone it if internet is enabled.

In [ ]:
import os, sys, subprocess, pathlib

REPO = pathlib.Path('/kaggle/input/medlift3d')     # <-- adjust
if not REPO.exists():
    REPO = pathlib.Path('/kaggle/working/medlift3d')
    if not REPO.exists():
        # Needs internet enabled; otherwise attach the repo as a dataset.
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/foyeznaeem/medlift3d.git', str(REPO)], check=True)

# Kaggle input is read-only, so work from a writable copy.
WORK = pathlib.Path('/kaggle/working')
if str(REPO).startswith('/kaggle/input'):
    subprocess.run(['cp', '-r', str(REPO), str(WORK / 'medlift3d')], check=True)
    REPO = WORK / 'medlift3d'

sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)

import torch
print('repo   :', REPO)
print('torch  :', torch.__version__, '| cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  gpu {i}: {p.name}  {p.total_memory/2**30:.1f} GiB')

## 1 · Gates — run these before spending any GPU time

| Gate | Asserts | Guards against |
|---|---|---|
| G1 | one canonical grid, affine never dropped | comparing volumes on different grids |
| G2 | `fp` differentiable, `bp` its exact adjoint | a projection loss with **no gradient** |
| G3 | water cylinder integral = `mu_water·2R` | HU/density confusion, scale errors |
| G4 | a *perfect* reconstruction scores perfectly | metrics that cannot detect anything |

If any gate fails, stop. Every one of these failures is silent and invalidates
every downstream number.

In [ ]:
!python scripts/run_gates.py

## 2 · Data

Three routes. Run exactly **one** of 2a / 2b / 2c per session — whichever cell
you run sets `DATA`, and every cell from here on reads that variable rather
than hardcoding a path.

**2a — synthetic phantoms (default).** No download, so the whole pipeline is
verifiable immediately. Drop to `--shape 128 128 128` if you are tight on
time; the paper target is 256³ @ 1.5 mm.

**2b — LUNA16 (real CT, no nodule masks).** Attach a Kaggle-hosted LUNA16
mirror (e.g. `avc0706/luna16`) as a dataset. Real anatomy for training the
prior and for reconstruction, but `mdvc.py` and `hallucination.py` read
`meta["nodules"]` and silently skip any case without it — this route supports
global fidelity metrics (PSNR/SSIM/MAE) only, not nodule Dice/volume error.

**2c — LIDC-IDRI via pylidc (real CT + four-reader nodule contours).** Attach
the full LIDC-IDRI DICOM mirror (e.g.
`justinkirby/the-cancer-imaging-archive-lidcidri`), turn internet **On** in
Settings, `pip install pylidc`. The only route that supports Dice, volume
error, MDVC and the hallucination audit on real data. Two cells: 2c-i sets up
pylidc and verifies the mirror actually carries the XML annotations *before*
any GPU time is spent; 2c-ii ingests the cases.


In [ ]:
DATA = '/kaggle/working/data/phantom'

!python scripts/make_phantoms.py \
    --n-cases 60 \
    --shape 256 256 256 --spacing 1.5 1.5 1.5 \
    --views-a 16 32 --views-b 15 \
    --nodules 3 --device cuda \
    --out {DATA}


### 2b · LUNA16 — skip if you ran 2a or will run 2c

Attach the dataset first (Add Input → Datasets → search "luna16"), then adjust
`LUNA_DIR` to match the slug it mounts at.


In [ ]:
DATA = '/kaggle/working/data/lidc'
LUNA_DIR = '/kaggle/input/luna16'          # <-- adjust to the attached dataset

!python scripts/prepare_lidc.py --source dir \
    --in {LUNA_DIR} \
    --out {DATA} \
    --limit 60


### 2c · LIDC-IDRI via pylidc — skip if you ran 2a or 2b

Attach the full LIDC-IDRI DICOM mirror (Add Input → Datasets → search
"cancer imaging archive lidc"), then turn on **Settings → Internet** for this
notebook (off by default only for competition-locked submissions; a normal
notebook can enable it). `LIDC_ROOT` must match wherever Kaggle mounts it.


In [ ]:
# Settings -> Internet: On, first. Skip the install if pylidc is already on the image.
%pip install -q pylidc

import pathlib

LIDC_ROOT = pathlib.Path('/kaggle/input/the-cancer-imaging-archive-lidcidri')  # <-- adjust
WAREHOUSE = pathlib.Path('/kaggle/working/pylidc_warehouse')                  # must be writable
WAREHOUSE.mkdir(parents=True, exist_ok=True)

pathlib.Path.home().joinpath('.pylidcrc').write_text(
    f'[dicom]\npath = {LIDC_ROOT}\nwarehouse_path = {WAREHOUSE}\n')

# pylidc needs the per-series XML annotation alongside the DICOM images.
# Check that BEFORE spending any GPU time on a mirror that turns out to be
# images-only -- an images-only mirror still runs pylidc without error, it
# just clusters zero annotations and every case comes back with no nodules,
# indistinguishable from route 2b except three hours later.
n_dcm = sum(1 for _ in LIDC_ROOT.rglob('*.dcm'))
n_xml = sum(1 for _ in LIDC_ROOT.rglob('*.xml'))
print(f'{n_dcm} DICOM files, {n_xml} XML annotation files under {LIDC_ROOT}')
assert n_dcm > 0, 'no DICOM found -- check LIDC_ROOT matches the attached dataset'
assert n_xml > 0, 'no XML annotations -- this mirror cannot support pylidc; use 2b instead'


In [ ]:
DATA = '/kaggle/working/data/lidc'

# pylidc builds its case index from scratch on first touch, so start small and
# confirm "M with nodule masks" is close to N before raising --limit.
!python scripts/prepare_lidc.py --source pylidc \
    --limit 40 --max-slice-thickness 1.5 \
    --out {DATA}


## 3 · Train the prior

~48M params at 256², batch 8 with AMP ≈ 7 GB on a T4.

**Rerun this cell in a new session to resume** — it picks up from `last.pt`
with the optimiser, scaler, step and best-val intact. `--max-hours 11` stops
cleanly before Kaggle kills the session, so no progress is ever lost to the cap.

In [ ]:
!python scripts/train_prior.py \
    --data {DATA} \
    --out  /kaggle/working/runs/prior \
    --epochs 60 --batch-size 8 --base-dim 64 \
    --amp --workers 2 --max-hours 11

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
log = pd.read_csv('/kaggle/working/runs/prior/log.csv')
ax = log.plot(x='epoch', y=['train_loss', 'val_loss'], figsize=(6, 3.4), grid=True)
ax.set_ylabel('eps-prediction MSE'); plt.tight_layout(); plt.show()
log.tail()

## 4 · Reconstruct

Baselines first — a learned prior has to beat them to justify its existence.
`--n-posterior 8` gives the mean reconstruction *and* the per-voxel uncertainty
map; it costs ~270 MB at 256³ and is the only thing that lets a reader tell
measured structure from structure the prior invented.

In [ ]:
# DATA was set in section 2 above (whichever of 2a/2b/2c you ran)
RECON = '/kaggle/working/runs/recon'
PRIOR = '/kaggle/working/runs/prior/best.pt'

for method in ['fbp', 'sirt_tv', 'cgls']:
    !python scripts/reconstruct.py --data {DATA} --out {RECON} \
        --track A16 --method {method} --n-iter 60 --device cuda

!python scripts/reconstruct.py --data {DATA} --out {RECON} \
    --track A16 --method diffusion --prior {PRIOR} \
    --n-steps 50 --n-posterior 8 --slice-batch 16 --device cuda

## 5 · Evaluate

Paired metrics, because this is a paired per-patient reconstruction task with
ground truth for every case — not FID/MMD, which measure distributional
similarity for *unconditional* generation. Nodule metrics are computed inside a
dilated bounding box, and PSNR/SSIM inside the lung mask.

In [ ]:
!python scripts/evaluate.py --recon {RECON} --data {DATA}

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob(f'{RECON}/*/*.png'))[:6]:
    print(f.split('/')[-2], '·', f.split('/')[-1]); display(Image(f))

## 6 · The experiments that make the contribution

The architecture alone is not the novelty — R²-Gaussian, X-Gaussian, DOLCE and
DiffusionMBIR already occupy that ground. These three are:

1. **MDVC** — minimum detectable volume change vs view count and arc. Answers the
   Volume Doubling Time objective directly, and says where absolute volumetry
   stops being trustworthy.
2. **Hallucination audit** — insert / erase / present. Yields detection
   sensitivity and the **false-positive nodule rate**, which is the number a
   clinician cares about given the 96% LDCT false-positive rate.
3. **O5 ablation** — does the Gaussian parameterisation earn its place? A negative
   answer is a legitimate result.

> Needs nodule masks. If `DATA` came from **2a** or **2c** you have them; from **2b** (LUNA16) you do not, and both `mdvc.py` and `hallucination.py` will silently produce zero rows -- every case is skipped, not an error.


In [ ]:
!python scripts/mdvc.py --data {DATA} --out /kaggle/working/runs/mdvc \
    --tracks A32 A16 B15 --methods sirt_tv diffusion --prior {PRIOR} \
    --gains 0.05 0.10 0.20 0.30 0.50 --limit 5 --device cuda

In [ ]:
display(Image('/kaggle/working/runs/mdvc/mdvc.png'))

In [ ]:
!python scripts/hallucination.py --data {DATA} \
    --out /kaggle/working/runs/hallucination \
    --tracks A16 B15 --methods sirt_tv diffusion --prior {PRIOR} \
    --limit 5 --device cuda

In [ ]:
!python scripts/ablate_roi.py --data {DATA} \
    --out /kaggle/working/runs/ablate_roi \
    --track A16 --limit 3 --iters 800 --device cuda

## 7 · Collect outputs

Kaggle persists `/kaggle/working` (~20 GB). Checkpoints are large, so keep the
one you need and drop the rest before committing the notebook.

In [ ]:
import shutil, pathlib

for p in pathlib.Path('/kaggle/working/runs/prior').glob('last.pt'):
    print('consider deleting to save space:', p, f'{p.stat().st_size/2**20:.0f} MiB')

shutil.make_archive('/kaggle/working/medlift3d_results', 'zip',
                    '/kaggle/working/runs', )
print('bundled -> /kaggle/working/medlift3d_results.zip')